# Clean and load `application`

Aroa's table. Loads the already-trimmed columns (from
`column_selection_combined.ipynb`), cleans nulls/types/duplicates, and
loads the result into MySQL.

In [ ]:
import pandas as pd
import yaml
import getpass
from urllib.parse import quote_plus
from sqlalchemy import create_engine

config = yaml.safe_load(open("../config.yaml"))

## Load

In [2]:
application = pd.read_csv(config["input_data"]["application"])
application.shape

(307511, 122)

In [3]:
application_columns = [
    "SK_ID_CURR", "TARGET",
    "CODE_GENDER", "CNT_CHILDREN", "CNT_FAM_MEMBERS",
    "NAME_FAMILY_STATUS", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY",
    "NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE", "OCCUPATION_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_HOUSING_TYPE",
    "DAYS_BIRTH", "DAYS_EMPLOYED"
]
application = application[application_columns].copy()
application.shape

(307511, 18)

## Explore

In [4]:
application.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   SK_ID_CURR           307511 non-null  int64  
 1   TARGET               307511 non-null  int64  
 2   CODE_GENDER          307511 non-null  object 
 3   CNT_CHILDREN         307511 non-null  int64  
 4   CNT_FAM_MEMBERS      307509 non-null  float64
 5   NAME_FAMILY_STATUS   307511 non-null  object 
 6   FLAG_OWN_CAR         307511 non-null  object 
 7   FLAG_OWN_REALTY      307511 non-null  object 
 8   AMT_INCOME_TOTAL     307511 non-null  float64
 9   AMT_CREDIT           307511 non-null  float64
 10  AMT_ANNUITY          307499 non-null  float64
 11  NAME_CONTRACT_TYPE   307511 non-null  object 
 12  NAME_INCOME_TYPE     307511 non-null  object 
 13  OCCUPATION_TYPE      211120 non-null  object 
 14  NAME_EDUCATION_TYPE  307511 non-null  object 
 15  NAME_HOUSING_TYPE

In [5]:
application.isna().sum()

SK_ID_CURR                 0
TARGET                     0
CODE_GENDER                0
CNT_CHILDREN               0
CNT_FAM_MEMBERS            2
NAME_FAMILY_STATUS         0
FLAG_OWN_CAR               0
FLAG_OWN_REALTY            0
AMT_INCOME_TOTAL           0
AMT_CREDIT                 0
AMT_ANNUITY               12
NAME_CONTRACT_TYPE         0
NAME_INCOME_TYPE           0
OCCUPATION_TYPE        96391
NAME_EDUCATION_TYPE        0
NAME_HOUSING_TYPE          0
DAYS_BIRTH                 0
DAYS_EMPLOYED              0
dtype: int64

In [6]:
application.duplicated(subset="SK_ID_CURR").sum()

np.int64(0)

## Clean

- `CODE_GENDER`: 4 rows have `'XNA'` -- can't classify, negligible, drop.
- `CNT_FAM_MEMBERS`: 2 null rows -- negligible, drop.
- `AMT_ANNUITY`: 12 null rows -- negligible, drop.
- `OCCUPATION_TYPE`: 96,391 nulls, but 100% of them are `Pensioner` or
  `Unemployed` in `NAME_INCOME_TYPE` -- this isn't missing data, it's
  structural (retired/unemployed people have no occupation). Fill with
  `"Not Employed"` instead of dropping 31% of the table.
- `DAYS_EMPLOYED`: 55,374 rows have the placeholder value `365243`, for the
  exact same Pensioner/Unemployed population -- a known bug in this Kaggle
  dataset. Replace with `NaN` so it can't skew any calculation.

In [7]:
# CODE_GENDER: drop the 4 'XNA' rows -- can't classify, negligible
application = application[application["CODE_GENDER"] != "XNA"]

# CNT_FAM_MEMBERS: drop the 2 null rows -- negligible
application = application.dropna(subset=["CNT_FAM_MEMBERS"])

# AMT_ANNUITY: drop the 12 null rows -- negligible
application = application.dropna(subset=["AMT_ANNUITY"])

# OCCUPATION_TYPE: null means "not employed" (100% of Pensioner/Unemployed),
# not missing data -- fill instead of dropping 31% of the table
application["OCCUPATION_TYPE"] = application["OCCUPATION_TYPE"].fillna("Not Employed")

# DAYS_EMPLOYED: 365243 is a known placeholder bug in this dataset for the
# same not-employed population -- replace with NaN so it can't skew any mean
application["DAYS_EMPLOYED"] = application["DAYS_EMPLOYED"].replace(365243, pd.NA)

application.shape

(307493, 18)

In [8]:
# Verify: nulls should be gone except DAYS_EMPLOYED (left as NaN on purpose)
application.isna().sum()

SK_ID_CURR                 0
TARGET                     0
CODE_GENDER                0
CNT_CHILDREN               0
CNT_FAM_MEMBERS            0
NAME_FAMILY_STATUS         0
FLAG_OWN_CAR               0
FLAG_OWN_REALTY            0
AMT_INCOME_TOTAL           0
AMT_CREDIT                 0
AMT_ANNUITY                0
NAME_CONTRACT_TYPE         0
NAME_INCOME_TYPE           0
OCCUPATION_TYPE            0
NAME_EDUCATION_TYPE        0
NAME_HOUSING_TYPE          0
DAYS_BIRTH                 0
DAYS_EMPLOYED          55374
dtype: int64

## Save clean CSV

In [9]:
application.to_csv(config["output_data"]["application"], index=False)

## Load into MySQL

In [ ]:
db = config["database"]
password = getpass.getpass("MySQL password (press Enter if none): ")
engine = create_engine(f"mysql+pymysql://{db['user']}:{quote_plus(password)}@{db['host']}/{db['name']}")

application.to_sql(
    "application",
    engine,
    if_exists="append",
    index=False,
    chunksize=10000,
    method="multi",
)